# Chapter 44 — Appendix B — The Terminal Interface, in Detail

**Companion to *Pi Agents*** · [`Pi Agents` chapter 44](https://programmer.ie/books/pi/44-chapter/)

| | |
|---|---|
| Chapter | `44-chapter.md` · weight 44 · `/books/pi/44-chapter/` |
| Notebook | `notebooks/pi/44-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Which of the six layers can be checked without a terminal, and which genuinely
cannot?

The chapter's point: the terminal UI is a **layered system** — terminal emulator,
reporting protocols, pi-tui components, interactive mode, configuration,
extensions. Layers 3–6 are checkable headlessly; layers 1–2 need a real
terminal that reports capabilities and keys.


## What this notebook establishes

- **Layer 3 (pi-tui) is checkable headlessly**: `visibleWidth` counts display columns for wide characters, emoji and ANSI; `truncateToWidth`, `wrapTextWithAnsi`, `sliceByColumn` do the width work; `CURSOR_MARKER` and `KeybindingsManager` exist and behave.
- **Layer 6 (extensions) is checkable**: the chapter 20 component and mode-gate tests run without a terminal.
- **Theme colour parsing is headless**: `parseColor` accepts a hex and rejects an invalid value.
- **Layers 1–2 are NOT checkable without a terminal** — Shift+Enter reporting, Kitty keyboard negotiation, and hardware-cursor placement are manual checks.

## What this notebook does **not** establish

- **Layers 1–2 are not executable** without the actual terminal: Shift+Enter reporting, Kitty keyboard negotiation, tmux passthrough, per-terminal settings, hardware cursor and IME placement, the ≤100 ms theme timeout.
- **`examples/probe/` is unverified scratch, not evidence** — this notebook does not rely on it.
- **A capability-override table is declared, not measured here.**


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** (layers 3 and 6 headless) + **Manual/optional live** (layers 1–2, a NOT RUN checklist).
* **Evidence scope:** `in_process_runtime` + `declarations`. No terminal is attached.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(44))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/44-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the pandoc-style width rules at layer 3

The pi-tui width functions are pure and headless. Wide characters and emoji
count in display columns, not string length; ANSI escapes are stripped.

In [3]:
TUI = pinb.driver_source("ch44-appendix.ts")
tui = pinb.driver(TUI, label="ch44-appendix", timeout=120)
w = tui["visibleWidth"]
print(pinb.md_table(
    ["input", "visibleWidth"],
    [[repr("hello"), w["plain"]],
     [repr("路径匹配"), w["wide"]],
     [repr("a😀b"), w["emoji"]],
     [repr("ANSI red text"), w["ansi"]]],
))

| input | visibleWidth |
|---|---|
| 'hello' | 5 |
| '路径匹配' | 8 |
| 'a😀b' | 4 |
| 'ANSI red text' | 8 |


In [4]:
pinb.show_checks([
    pinb.check("wide characters count as 2 columns",
               w["wide"] == 8, f"{w['wide']} columns for 4 CJK chars"),
    pinb.check("emoji count wider than one column",
               w["emoji"] > 2, f"{w['emoji']} columns"),
    pinb.check("ANSI escapes are stripped: red text is 8 columns",
               w["ansi"] == 8, f"{w['ansi']} columns"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  -------------------------
PASS  wide characters count as 2 columns  8 columns for 4 CJK chars
PASS  emoji count wider than one column  4 columns
PASS  ANSI escapes are stripped: red text is 8 columns  8 columns

3/3 assertions held.


## The width helpers: truncate, wrap, slice

In [5]:
tr = tui["truncate"]
print(pinb.md_table(
    ["operation", "result"],
    [["truncate abcdef at 3", tr["at3"]],
     ["truncate with ellipsis", tr["at3Ellipsis"]],
     ["truncate does nothing when it fits", tr["noTrunc"]],
     ["wrap at 10 columns: lines", tui["wrap"]["count"]],
     ["slice columns 2..5 of abcdef", tui["slice"]],
     ["CURSOR_MARKER present", "yes" if tui["cursorMarker"]["exists"] else "no"],
     ["KeybindingsManager constructible", "yes" if tui["keybindings"]["managerConstructible"] is True else "no"]],
))

| operation | result |
|---|---|
| truncate abcdef at 3 | ... |
| truncate with ellipsis | ab… |
| truncate does nothing when it fits | abc |
| wrap at 10 columns: lines | 5 |
| slice columns 2..5 of abcdef | cde |
| CURSOR_MARKER present | yes |
| KeybindingsManager constructible | yes |


In [6]:
pinb.show_checks([
    pinb.check("truncate changes the text when it does not fit",
               tui["truncate"]["at3"] != "abcdef" and "..." in tui["truncate"]["at3"], "default ellipsis applied"),
    pinb.check("truncate can take a custom ellipsis",
               "…" in tui["truncate"]["at3Ellipsis"], "ellipsis used"),
    pinb.check("truncate leaves text alone when it fits",
               tui["truncate"]["noTrunc"] == "abc", "unchanged"),
    pinb.check("wrap splits a long line into width-sized lines",
               tui["wrap"]["count"] > 1, f"{tui['wrap']['count']} lines"),
    pinb.check("sliceByColumn works on columns, not characters",
               tui["slice"] == "cde", "cde"),
    pinb.check("CURSOR_MARKER is a real marker string",
               tui["cursorMarker"]["exists"], tui["cursorMarker"]["value"]),
    pinb.check("KeybindingsManager can be constructed headlessly",
               tui["keybindings"]["managerConstructible"] is True, "constructible"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------------  ------------------------
PASS  truncate changes the text when it does not fit  default ellipsis applied
PASS  truncate can take a custom ellipsis  ellipsis used
PASS  truncate leaves text alone when it fits  unchanged
PASS  wrap splits a long line into width-sized lines  5 lines
PASS  sliceByColumn works on columns, not characters  cde
PASS  CURSOR_MARKER is a real marker string  _pi:c
PASS  KeybindingsManager can be constructed headlessly  constructible

7/7 assertions held.


## Layer 6: the mode gate and the component from chapter 20

The chapter 20 UI tests run headless: the `hasUI` vs `mode === "tui"` distinction
and the `BlockListComponent` width cache. They are this layer's checkable row.

## The layer-6 checks (component and mode gate)

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [7]:
PATTERNS = 'ch20-ui/ui.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch20-ui/ui.test.ts', show="component")

$ node --test ch20-ui/ui.test.ts
  PASS  7 passed, 0 failed, 0 skipped  in 917 ms

  tests matching 'component':
    ok   component: every line fits the width, measured in columns, including wide characters
    ok   component: output is cached by width and recomputed after invalidate()
    ok   component: it finishes through the completion callback on q or Escape


## Layers 1–2: the NOT RUN checklist

These genuinely cannot be checked without the terminal Pi is talking to. This
notebook says so instead of pretending.

In [8]:
pinb.unrun("Shift+Enter and modified-Enter reporting", "layer 2 needs a terminal that reports the modified key")
pinb.unrun("Kitty keyboard protocol negotiation", "layer 2 needs the terminal's protocol answer")
pinb.unrun("tmux passthrough and per-terminal settings", "layer 2 needs the terminal session")
pinb.unrun("hardware cursor and IME placement", "layer 4 needs an interactive terminal")
pinb.unrun("the <= 100 ms theme-timeout behaviour", "layer 2 needs the terminal's answer to arrive late")
print()
print("The capability-override table is declared in the chapter; it is not measured here.")

NOT RUN - Shift+Enter and modified-Enter reporting
  reason: layer 2 needs a terminal that reports the modified key
  No output is shown because none was produced. This cell makes no claim.
NOT RUN - Kitty keyboard protocol negotiation
  reason: layer 2 needs the terminal's protocol answer
  No output is shown because none was produced. This cell makes no claim.
NOT RUN - tmux passthrough and per-terminal settings
  reason: layer 2 needs the terminal session
  No output is shown because none was produced. This cell makes no claim.
NOT RUN - hardware cursor and IME placement
  reason: layer 4 needs an interactive terminal
  No output is shown because none was produced. This cell makes no claim.
NOT RUN - the <= 100 ms theme-timeout behaviour
  reason: layer 2 needs the terminal's answer to arrive late
  No output is shown because none was produced. This cell makes no claim.

The capability-override table is declared in the chapter; it is not measured here.


## Your turn: predict, then run

**Predict first.** `visibleWidth("路径匹配")` returned 8. What does the length —
not the width — return in JavaScript?

**Then change one input.** `sliceByColumn("abcdef", 2, 3)` returned `"cde"`.
What does `sliceByColumn("a😀bc", 1, 2)` return — by columns, not characters?

**Predict the boundary.** `CURSOR_MARKER` is `ESC-pi:c-BEL`. Where does the TUI
use it, and why would a component that forgets it break IME?

In [9]:
print("Predictions:")
print("  1. JS length of 路径匹配 is 4 - width 8, length 4.")
print("  2. sliceByColumn(a😀bc, 1, 2): 2 columns starting col 1 - the emoji is 2 wide.")
print("  3. CURSOR_MARKER positions the hardware cursor for IME; forgetting it misplaces candidates.")
print()
print("Observed above:")
print(f"  visibleWidth: plain={w['plain']}, wide={w['wide']}, emoji={w['emoji']}, ansi={w['ansi']}")
print(f"  truncate at 3: {tui['truncate']['at3']!r}")
print(f"  wrap count: {tui['wrap']['count']}, slice: {tui['slice']!r}")
print()
assert w["wide"] == 8 and w["ansi"] == 8
assert tui["truncate"]["noTrunc"] == "abc"
assert tui["wrap"]["count"] > 1 and tui["slice"] == "cde"
assert tui["cursorMarker"]["exists"]
print("held: layer 3 width rules are headless; layers 1-2 are NOT RUN; CURSOR_MARKER exists")

Predictions:
  1. JS length of 路径匹配 is 4 - width 8, length 4.
  2. sliceByColumn(a😀bc, 1, 2): 2 columns starting col 1 - the emoji is 2 wide.
  3. CURSOR_MARKER positions the hardware cursor for IME; forgetting it misplaces candidates.

Observed above:
  visibleWidth: plain=5, wide=8, emoji=4, ansi=8
  truncate at 3: '\x1b...\x1b'
  wrap count: 5, slice: 'cde'

held: layer 3 width rules are headless; layers 1-2 are NOT RUN; CURSOR_MARKER exists


## Interpretation

The six layers, and what this notebook can check:

| Layer | Checkable here? | What was done |
|---|---|---|
| 1. Terminal emulator | **No** | NOT RUN — needs the terminal |
| 2. Reporting protocols | **No** | NOT RUN — needs the protocol answer |
| 3. pi-tui components | **Yes** | `visibleWidth`, `truncateToWidth`, `wrapTextWithAnsi`, `sliceByColumn`, `CURSOR_MARKER`, `KeybindingsManager` |
| 4. Interactive mode | Mostly not | NOT RUN for cursor/IME; the rest needs a terminal |
| 5. Your configuration | Partial | Keybindings manager constructible; theme colour parsing |
| 6. Your extensions | **Yes** | Chapter 20's component and mode-gate tests |

The practical rules:
1. **Width is columns, not length** — wide characters and emoji prove it.
2. **`CURSOR_MARKER` is the IME hook** — a component that forgets it misplaces candidates.
3. **Layers 1–2 need the real terminal** — a headless run cannot see a key it never received.
4. **An unmeasured capability table stays a declaration** — this notebook does not pretend otherwise.
5. **`examples/probe/` is scratch, not evidence** — nothing here relies on it.

## Sources, execution mode and limitations

**Execution mode:** **Run** (layers 3 and 6 headless) + **Manual/optional live** (layers 1–2, a NOT RUN checklist).

**Evidence scope:** `in_process_runtime` + `declarations`. No terminal is attached.

### What was read or run

- **Ran** `drivers/ch44-appendix.ts`: `visibleWidth`, `truncateToWidth`, `wrapTextWithAnsi`, `sliceByColumn`, `CURSOR_MARKER`, `KeybindingsManager`, `parseColor` — all headless.
- **Ran** `ch20-ui/ui.test.ts` (7 tests) for the layer-6 component and mode-gate checks.
- **Read** the pinned `@earendil-works/pi-tui` declarations.

### Limitations

- **Layers 1–2 are not executable** without the actual terminal — the NOT RUN checklist states the prerequisites.
- **A capability-override table is declared, not measured here.**
- **`examples/probe/` is unverified scratch, not evidence.**

### Optional work that was NOT run

- Shift+Enter and modified-Enter reporting (layer 2).
- Kitty keyboard negotiation, tmux passthrough, per-terminal settings (layer 2).
- Hardware cursor and IME placement (layer 4).
- The ≤100 ms theme-timeout behaviour (layer 2).

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
